In [ ]:
MODEL_DIR    = "/content/drive/MyDrive/mask2former-paintings-v3-ext"
DATASET_DIR  = "/content/drive/MyDrive/full_dataset"
OUT_DIR      = "/content/drive/MyDrive/composition_v2"
PREVIEW_EVERY = 20          # save a colourised mask every Nth painting; None to disable
MAX_SIDE      = None        # cap the long side before scoring; None = native resolution
ROW_CHUNK     = 512         # rows per chunk when accumulating (controls peak memory)
SEED          = 42

os.makedirs(OUT_DIR, exist_ok=True)
if PREVIEW_EVERY:
    os.makedirs(os.path.join(OUT_DIR, "preview_masks"), exist_ok=True)

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ==============================================================================
# MODEL
# ==============================================================================
processor = Mask2FormerImageProcessor.from_pretrained(MODEL_DIR)
model = Mask2FormerForUniversalSegmentation.from_pretrained(MODEL_DIR).to(device).eval()

id2label = {int(k): v for k, v in model.config.id2label.items()}
CLASS_IDS   = sorted(id2label)
CLASS_NAMES = [id2label[i] for i in CLASS_IDS]
N_CLASSES   = len(CLASS_IDS)
print(f"{N_CLASSES} classes: {CLASS_NAMES}")

# resolve the two excluded classes BY NAME, case-insensitively
def find_class(*candidates):
    low = {n.lower(): i for i, n in id2label.items()}
    for c in candidates:
        if c.lower() in low:
            return low[c.lower()]
    raise KeyError(f"none of {candidates} found in id2label: {CLASS_NAMES}")

SKY_ID = find_class("sky")
BG_ID  = find_class("background", "bg")
print(f"sky id = {SKY_ID}, background id = {BG_ID}")

# ==============================================================================
# SCORING
# ==============================================================================
@torch.no_grad()
def semantic_probabilities(image):
    """Return per-pixel class probabilities [C, H, W] at the image's native size.

    Reproduces Mask2FormerImageProcessor.post_process_semantic_segmentation
    (einsum of query class softmax and mask sigmoid, then resize), but keeps the
    full score map instead of collapsing it to argmax, and normalises it across
    classes so it can be read as a probability.
    """
    W, H = image.size
    inputs = processor(images=image, return_tensors="pt").to(device)
    out = model(**inputs)

    masks_classes = out.class_queries_logits.softmax(dim=-1)[..., :-1]   # [1, Q, C]
    masks_probs   = out.masks_queries_logits.sigmoid()                   # [1, Q, h, w]
    seg = torch.einsum("bqc,bqhw->bchw", masks_classes, masks_probs)     # [1, C, h, w]
    seg = F.interpolate(seg, size=(H, W), mode="bilinear", align_corners=False)
    seg = seg.clamp_min(1e-12)
    probs = seg / seg.sum(dim=1, keepdim=True)                           # [1, C, H, W]
    return probs[0]

def accumulate(probs):
    """Hard counts, soft (expected) counts and analytic soft variances.

    soft_count[c]    = sum_i p_ic                    (expected pixels of class c)
    soft_var[c]      = sum_i p_ic (1 - p_ic)         (Bernoulli variance, INDEPENDENT
                                                      pixels -- a lower bound, see note)
    Pixels in an image are strongly spatially autocorrelated, so soft_var
    understates the true uncertainty. The analysis script inflates it by an
    explicit, reported factor rather than pretending the bound is exact.
    """
    C, H, W = probs.shape
    hard = torch.zeros(C, dtype=torch.long, device=probs.device)
    soft = torch.zeros(C, dtype=torch.float64, device=probs.device)
    var  = torch.zeros(C, dtype=torch.float64, device=probs.device)
    conf_sum = 0.0
    for r0 in range(0, H, ROW_CHUNK):
        blk = probs[:, r0:r0 + ROW_CHUNK, :]                 # [C, h, W]
        am  = blk.argmax(dim=0)
        hard += torch.bincount(am.reshape(-1), minlength=C)
        b64  = blk.double()
        soft += b64.sum(dim=(1, 2))
        var  += (b64 * (1.0 - b64)).sum(dim=(1, 2))
        conf_sum += blk.max(dim=0).values.double().sum().item()
    return (hard.cpu().numpy(), soft.cpu().numpy(), var.cpu().numpy(),
            conf_sum / (H * W))

# CVAT palette, so predicted masks render in the same colours as the ground truth
# and can be placed side by side in a figure (examiner issue 8 asks for exactly this)
CVAT_COLORS = {
    0:  (0, 0, 0),        1:  (110, 13, 13),   2:  (96, 66, 7),
    3:  (131, 224, 112),  4:  (240, 120, 240), 5:  (37, 70, 103),
    6:  (230, 209, 168),  7:  (184, 61, 245),  8:  (65, 93, 125),
    9:  (48, 173, 48),    10: (18, 206, 242),  11: (13, 135, 53),
    12: (135, 246, 171),  13: (253, 164, 5),   14: (85, 144, 203),
}
assert set(CVAT_COLORS) == set(CLASS_IDS), (
    f"CVAT_COLORS keys {sorted(CVAT_COLORS)} != model ids {CLASS_IDS}")
PALETTE = np.zeros((max(CLASS_IDS) + 1, 3), dtype=np.uint8)
for i, c in CVAT_COLORS.items():
    PALETTE[i] = c

# ==============================================================================
# BATCH
# ==============================================================================
EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff")
images = sorted(f for f in os.listdir(DATASET_DIR) if f.lower().endswith(EXTS))
print(f"{len(images)} paintings found.\n")

rows, failures = [], []
for idx, name in enumerate(images, 1):
    path = os.path.join(DATASET_DIR, name)
    try:
        img = Image.open(path).convert("RGB")
        if MAX_SIDE and max(img.size) > MAX_SIDE:
            s = MAX_SIDE / max(img.size)
            img = img.resize((round(img.size[0]*s), round(img.size[1]*s)), Image.BICUBIC)
        W, H = img.size
        probs = semantic_probabilities(img)
        hard, soft, var, conf = accumulate(probs)

        total = int(hard.sum())
        terr_hard = total - int(hard[SKY_ID]) - int(hard[BG_ID])
        terr_soft = float(soft.sum() - soft[SKY_ID] - soft[BG_ID])

        rec = {
            "Image_Name": name,
            "width": W, "height": H,
            "total_pixels": total,
            "terrestrial_pixels_hard": terr_hard,
            "terrestrial_pixels_soft": terr_soft,
            "mean_pixel_confidence": conf,
        }
        for i, cid in enumerate(CLASS_IDS):
            cn = id2label[cid]
            rec[f"n_{cn}"]        = int(hard[i])    # integer pixel count, exact
            rec[f"soft_{cn}"]     = float(soft[i])  # expected pixel count
            rec[f"softvar_{cn}"]  = float(var[i])   # lower-bound variance
        rows.append(rec)

        if PREVIEW_EVERY and idx % PREVIEW_EVERY == 0:
            am = probs.argmax(0).cpu().numpy().astype(np.uint8)
            Image.fromarray(PALETTE[am]).save(
                os.path.join(OUT_DIR, "preview_masks", f"{os.path.splitext(name)[0]}_pred.png"))

        del probs
        if device.type == "cuda":
            torch.cuda.empty_cache()
        print(f"[{idx}/{len(images)}] {name}  terrestrial={terr_hard}  conf={conf:.3f}")

    except Exception as e:
        print(f"[ERROR] {name}: {e}")
        failures.append({"Image_Name": name, "error": str(e)})

# ==============================================================================
# EXPORT
# ==============================================================================
df = pd.DataFrame(rows)
csv_path = os.path.join(OUT_DIR, "composition_counts.csv")
df.to_csv(csv_path, index=False)          # CSV, not xlsx: no silent reformatting
print(f"\nwrote {csv_path}  ({len(df)} rows)")
if failures:
    pd.DataFrame(failures).to_csv(os.path.join(OUT_DIR, "failures.csv"), index=False)
    print(f"{len(failures)} failures written to failures.csv")

with open(os.path.join(MODEL_DIR, "config.json"), "rb") as f:
    cfg_hash = hashlib.sha256(f.read()).hexdigest()[:16]

prov = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model_dir": MODEL_DIR, "config_sha256_16": cfg_hash,
    "id2label": {str(k): v for k, v in id2label.items()},
    "sky_id": SKY_ID, "background_id": BG_ID,
    "n_images": len(df), "n_failures": len(failures),
    "max_side": MAX_SIDE, "seed": SEED,
    "python": platform.python_version(), "torch": torch.__version__,
    "transformers": transformers.__version__, "numpy": np.__version__,
    "pandas": pd.__version__,
    "cuda": torch.version.cuda, "device": str(device),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
with open(os.path.join(OUT_DIR, "run_provenance.json"), "w") as f:
    json.dump(prov, f, indent=2)
print("wrote run_provenance.json")

# quick integrity report
print("\nIntegrity check")
print(f"  hard counts sum to width*height for all rows: "
      f"{bool((df.total_pixels == df.width * df.height).all())}")
sc = df[[c for c in df.columns if c.startswith('soft_')]].sum(axis=1)
print(f"  soft counts sum to total_pixels (max abs dev): {float((sc - df.total_pixels).abs().max()):.2f}")
print(f"  mean per-pixel confidence: {df.mean_pixel_confidence.mean():.3f} "
      f"(min {df.mean_pixel_confidence.min():.3f})")


Device: cuda


Loading weights:   0%|          | 0/556 [00:00<?, ?it/s]

15 classes: ['background', 'building', 'earth', 'grass', 'animal', 'mountain', 'path_road', 'person', 'rock', 'shrub_bush', 'sky', 'tree_conical', 'tree_broadleaf', 'wooded_mass', 'Water']
sky id = 10, background id = 0
243 paintings found.

[1/243] 1.jpg  terrestrial=322419  conf=0.892
[2/243] 10.jpg  terrestrial=165384  conf=0.880
[3/243] 100.jpg  terrestrial=366391  conf=0.934
[4/243] 101.jpg  terrestrial=249118  conf=0.886
[5/243] 102.jpg  terrestrial=215484  conf=0.856
[6/243] 103.jpg  terrestrial=249474  conf=0.845
[7/243] 104.jpg  terrestrial=257281  conf=0.912
[8/243] 105.jpg  terrestrial=319757  conf=0.905
[9/243] 106.jpg  terrestrial=205938  conf=0.869
[10/243] 107.jpg  terrestrial=365453  conf=0.833
[11/243] 108.jpg  terrestrial=274365  conf=0.793
[12/243] 109.jpg  terrestrial=174017  conf=0.848
[13/243] 11.jpg  terrestrial=226175  conf=0.942
[14/243] 110.jpg  terrestrial=291884  conf=0.858
[15/243] 111.jpg  terrestrial=404142  conf=0.945
[16/243] 112.jpg  terrestrial=389857

In [ ]:
import os, json
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
from PIL import Image
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

MODEL_DIR = "/content/drive/MyDrive/mask2former-paintings-v3-ext"
IMG_DIR   = "/content/drive/MyDrive/TestData/TestImages"      # held-out paintings
GT_DIR    = "/content/drive/MyDrive/TestData/TestMasks"       # CVAT masks, class-index PNGs
OUT_DIR   = "/content/drive/MyDrive/composition_v2"
MIN_PAINTINGS = 5          # below this, a per-class metric is reported as unresolved
BENCHMARK_6   = []         # <- the 6 stems used for the cross-architecture comparison,
                           #    e.g. ["N-0109-00-000032-wpu", ...]. Leave empty to skip.
os.makedirs(OUT_DIR, exist_ok=True)

# Pooling used by the analysis -- must match analysis.R exactly
POOL = {
    "wooded":    ["wooded_mass", "tree_broadleaf", "tree_conical"],
    "open_veg":  ["grass", "shrub_bush"],
    "substrate": ["earth", "rock", "mountain"],
    "water":     ["Water"],
    "human":     ["building", "path_road", "person"],
}
EXCLUDE = ["sky", "background"]     # plus 'animal', handled below
DROP    = ["animal"]                # 3.2% mean cover, 43% zeros -- excluded from the
                                    # 5-part composition

# ------------------------------------------------------------------------------
# CVAT MASK DECODING
# ------------------------------------------------------------------------------
# CVAT exports RGB colour masks, not class-index images. Decoding by taking one
# channel is wrong -- it reads a colour value as a class ID. Every pixel must be
# matched against the exact palette below.
COLOR_MAP = {
    (0, 0, 0): 0,          # background
    (110, 13, 13): 1,      # building
    (96, 66, 7): 2,        # earth
    (131, 224, 112): 3,    # grass
    (240, 120, 240): 4,    # animal
    (37, 70, 103): 5,      # mountain
    (230, 209, 168): 6,    # path_road
    (184, 61, 245): 7,     # person
    (65, 93, 125): 8,      # rock
    (48, 173, 48): 9,      # shrub_bush
    (18, 206, 242): 10,    # sky
    (13, 135, 53): 11,     # tree_conical
    (135, 246, 171): 12,   # tree_broadleaf
    (253, 164, 5): 13,     # wooded_mass
    (85, 144, 203): 14,    # Water
}
NEAREST_TOL = 12      # max RGB distance for salvaging an off-palette pixel;
                      # set to 0 to refuse all inexact matches

_keys = np.array([(r << 16) | (g << 8) | b for (r, g, b) in COLOR_MAP], dtype=np.int64)
_vals = np.array(list(COLOR_MAP.values()), dtype=np.int16)
_ord  = np.argsort(_keys); _keys_s, _vals_s = _keys[_ord], _vals[_ord]
_palette_rgb = np.array(list(COLOR_MAP), dtype=np.int16)

def decode_mask(path):
    """RGB CVAT mask -> class-index array. Unmatched pixels become -1.

    Returns (labels, stats). Exact palette matching first; colours that are close
    to a palette entry (resampling or antialiasing artefacts) are snapped if
    within NEAREST_TOL, and counted. Anything further away stays -1 and is
    excluded from the confusion matrix rather than silently folded into
    background -- which would inflate class 0 and distort every metric.
    """
    a = np.array(Image.open(path).convert("RGB")).astype(np.int64)
    key = (a[..., 0] << 16) | (a[..., 1] << 8) | a[..., 2]
    pos = np.searchsorted(_keys_s, key)
    pos = np.clip(pos, 0, len(_keys_s) - 1)
    hit = _keys_s[pos] == key
    lab = np.where(hit, _vals_s[pos], -1).astype(np.int16)

    n_tot = lab.size
    n_bad = int((~hit).sum())
    n_snapped = 0
    if n_bad and NEAREST_TOL > 0:
        bad_rgb = a[~hit].reshape(-1, 3)
        uniq, inv = np.unique(bad_rgb, axis=0, return_inverse=True)
        dist = np.abs(uniq[:, None, :] - _palette_rgb[None, :, :]).sum(-1)
        nearest = dist.argmin(1); mind = dist.min(1)
        mapped = np.where(mind <= NEAREST_TOL, np.array(list(COLOR_MAP.values()))[nearest], -1)
        lab[~hit] = mapped[inv].astype(np.int16)
        n_snapped = int((mapped[inv] >= 0).sum())

    stats = dict(total=n_tot, off_palette=n_bad, snapped=n_snapped,
                 dropped=int((lab < 0).sum()))
    return lab, stats

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = Mask2FormerImageProcessor.from_pretrained(MODEL_DIR)
model = Mask2FormerForUniversalSegmentation.from_pretrained(MODEL_DIR).to(device).eval()
id2label = {int(k): v for k, v in model.config.id2label.items()}
IDS   = sorted(id2label); NAMES = [id2label[i] for i in IDS]; C = len(IDS)

# fail loudly if the CVAT palette and the model ontology disagree
assert set(COLOR_MAP.values()) == set(IDS), (
    f"COLOR_MAP indices {sorted(set(COLOR_MAP.values()))} != model ids {IDS}")
print("palette -> class mapping:")
for rgb, i in sorted(COLOR_MAP.items(), key=lambda kv: kv[1]):
    print(f"  {i:2d}  {str(rgb):18s} {id2label[i]}")

@torch.no_grad()
def predict(img, size_hw):
    inputs = processor(images=img, return_tensors="pt").to(device)
    o = model(**inputs)
    seg = torch.einsum("bqc,bqhw->bchw",
                       o.class_queries_logits.softmax(-1)[..., :-1],
                       o.masks_queries_logits.sigmoid())
    seg = F.interpolate(seg, size=size_hw, mode="bilinear", align_corners=False)
    return seg[0].argmax(0).cpu().numpy()

# ------------------------------------------------------------------ accumulate
pairs = []
for f in sorted(os.listdir(IMG_DIR)):
    stem = os.path.splitext(f)[0]
    for ext in (".png", ".tif", ".tiff"):
        g = os.path.join(GT_DIR, stem + ext)
        if os.path.exists(g):
            pairs.append((os.path.join(IMG_DIR, f), g, stem)); break
print(f"{len(pairs)} image/mask pairs")

per_painting = {}                     # for the bootstrap
M = np.zeros((C, C), dtype=np.int64)
decode_log = []
for ip, gp, stem in pairs:
    img = Image.open(ip).convert("RGB")
    gt, st = decode_mask(gp)
    pred = predict(img, gt.shape)
    m = np.zeros((C, C), dtype=np.int64)
    ok = gt >= 0
    np.add.at(m, (gt[ok].ravel().astype(np.int64), pred[ok].ravel().astype(np.int64)), 1)
    per_painting[stem] = m; M += m
    st["stem"] = stem
    st["pct_dropped"] = 100 * st["dropped"] / st["total"]
    decode_log.append(st)
    flag = "  <-- CHECK" if st["pct_dropped"] > 1 else ""
    print(f"  {stem}: {ok.sum():>9} px used | off-palette {st['off_palette']:>8} "
          f"| snapped {st['snapped']:>8} | dropped {st['pct_dropped']:.3f}%{flag}")

dec = pd.DataFrame(decode_log)
dec.to_csv(os.path.join(OUT_DIR, "mask_decode_log.csv"), index=False)
print(f"\nDecoding summary: {dec.pct_dropped.max():.3f}% dropped at worst, "
      f"{dec.pct_dropped.mean():.3f}% mean.")
if dec.pct_dropped.max() > 1:
    print("  >1% of pixels in some mask did not match the palette. Check whether the")
    print("  masks were ever resized with interpolation or saved as JPEG -- either")
    print("  introduces colours that are not in COLOR_MAP. Re-export as lossless PNG")
    print("  with nearest-neighbour resampling if so.")

pd.DataFrame(M, index=NAMES, columns=NAMES).to_csv(
    os.path.join(OUT_DIR, "confusion_15class_counts.csv"))

# per-painting matrices: required to bootstrap the correction matrix R at the level
# that actually carries uncertainty. Pixel-level noise is negligible (millions of
# pixels); painting-level noise is what matters with n = 16.
np.savez_compressed(os.path.join(OUT_DIR, "per_painting_confusion.npz"),
                    stems=np.array(list(per_painting)),
                    mats=np.stack([per_painting[k] for k in per_painting]),
                    names=np.array(NAMES))
print("wrote per_painting_confusion.npz")

# ------------------------------------------------------------------ per class
def metrics(m):
    """IoU, precision, recall for a square confusion matrix of ANY size.
    Sized from the matrix, not from the global class count -- this is called with
    both the 15x15 and the pooled 5x5 matrix."""
    m  = np.asarray(m, dtype=float)
    k  = m.shape[0]
    tp = np.diag(m)
    row, col = m.sum(1), m.sum(0)
    rec  = np.divide(tp, row, out=np.zeros(k), where=row > 0)
    prec = np.divide(tp, col, out=np.zeros(k), where=col > 0)
    den  = row + col - tp
    iou  = np.divide(tp, den, out=np.zeros(k), where=den > 0)
    return iou, prec, rec

iou, prec, rec = metrics(M)
per_class = pd.DataFrame({"class": NAMES, "IoU": 100*iou,
                          "precision": 100*prec, "recall": 100*rec,
                          "gt_pixels": M.sum(1), "pred_pixels": M.sum(0)})
print("\n15-class metrics (compare against your Table 1):")
print(per_class.round(2).to_string(index=False))

# ------------------------------------------------------------------ pooled 5x5
groups = list(POOL)
idx = {g: [NAMES.index(n) for n in POOL[g] if n in NAMES] for g in groups}
keep = sorted(set(sum(idx.values(), [])))
Mp = np.zeros((len(groups), len(groups)), dtype=np.int64)
for a, ga in enumerate(groups):
    for b, gb in enumerate(groups):
        Mp[a, b] = M[np.ix_(idx[ga], idx[gb])].sum()
pooled = pd.DataFrame(Mp, index=groups, columns=groups)
pooled.to_csv(os.path.join(OUT_DIR, "confusion_pooled_counts.csv"))

iou_p, prec_p, rec_p = metrics(Mp)
pooled_metrics = pd.DataFrame({"part": groups, "IoU": 100*iou_p,
                               "precision": 100*prec_p, "recall": 100*rec_p})
print("\nPOOLED-CATEGORY metrics -- this table answers examiner issue 8,")
print("which asks for accuracy at the level the ecology is actually reported at:")
print(pooled_metrics.round(2).to_string(index=False))

R = Mp / np.maximum(Mp.sum(1, keepdims=True), 1)      # P(pred = j | true = i)
np.savetxt(os.path.join(OUT_DIR, "R_pooled_rownorm.csv"), R, delimiter=",")
print(f"\ncondition number of R^T (5-part): {np.linalg.cond(R.T):.1f}")
R15 = M / np.maximum(M.sum(1, keepdims=True), 1)
print(f"condition number of R^T (15-class): {np.linalg.cond(R15.T):.1f}"
      "   <- if this is large, do not invert it; use the pooled system")

# ------------------------------------------------- measurement-error calibration
# The model's own soft variance implies the composition is known to about +/-0.04
# percentage points, which is off by orders of magnitude: mean per-pixel confidence
# is 0.905 while pooled IoU is far lower, i.e. the network is confidently wrong.
# So calibrate measurement error EMPIRICALLY: compare predicted against true pooled
# cover on each validation painting and use the observed discrepancy.
cal = []
for ip, gp, stem in pairs:
    img = Image.open(ip).convert("RGB")
    gt, _ = decode_mask(gp)
    pred  = predict(img, gt.shape)
    ok = gt >= 0
    row = {"stem": stem}
    gt_ok, pr_ok = gt[ok], pred[ok]
    # Denominator = the pooled parts ONLY. Anything not in a part (sky, background,
    # animal) must be excluded from BOTH numerator and denominator; leaving it in
    # makes the rows sum to less than 1 by a different amount in truth and
    # prediction, which fabricates error.
    part_ids = {g: np.array(idx[g]) for g in groups}
    in_parts = np.concatenate(list(part_ids.values()))
    m_gt = np.isin(gt_ok, in_parts); m_pr = np.isin(pr_ok, in_parts)
    for g in groups:
        row[f"true_{g}"] = np.isin(gt_ok[m_gt], part_ids[g]).mean() if m_gt.any() else np.nan
        row[f"pred_{g}"] = np.isin(pr_ok[m_pr], part_ids[g]).mean() if m_pr.any() else np.nan
    cal.append(row)
cal = pd.DataFrame(cal)
for g in groups:
    cal[f"err_{g}"] = cal[f"pred_{g}"] - cal[f"true_{g}"]
cal.to_csv(os.path.join(OUT_DIR, "cover_calibration.csv"), index=False)

# Error scales with how much of a part is present (corr(|err|, cover) reaches 0.83),
# so report a LOG-RATIO error as well -- that is the scale the analysis works on and
# the one a multiplicative measurement model needs.
eps = 1e-4
logerr = {g: np.log((cal[f"pred_{g}"] + eps) / (cal[f"true_{g}"] + eps)) for g in groups}
meas = pd.DataFrame({
    "part": groups,
    "bias_pp":  [100*cal[f"err_{g}"].mean() for g in groups],
    "sd_pp":    [100*cal[f"err_{g}"].std(ddof=1) for g in groups],
    "rmse_pp":  [100*np.sqrt((cal[f"err_{g}"]**2).mean()) for g in groups],
    "mean_true_pp": [100*cal[f"true_{g}"].mean() for g in groups],
    "log_bias": [logerr[g].mean() for g in groups],
    "log_sd":   [logerr[g].std(ddof=1) for g in groups],
    "corr_abserr_cover": [np.corrcoef(cal[f"true_{g}"], cal[f"err_{g}"].abs())[0,1]
                          for g in groups]})
meas.round(2).to_csv(os.path.join(OUT_DIR, "measurement_error.csv"), index=False)
print("\nMEASUREMENT ERROR, calibrated on the validation paintings")
print("(bias = systematic over/under-estimation; sd = per-painting scatter)")
print(meas.round(2).to_string(index=False))
print("\nFeed measurement_error.csv to analysis.R -- these are the real SDs for the")
print("imputation step. Do NOT use the model's soft variance for this; it is ~250x")
print("too small in SD terms because the network is confidently wrong.")

# ------------------------------------------------------- prevalence + subsets
prevalence = np.array([sum((m.sum(1)[i] > 0) for m in per_painting.values())
                       for i in range(C)])
per_class["paintings_present"] = prevalence
per_class["resolved"] = prevalence >= MIN_PAINTINGS
per_class.round(2).to_csv(os.path.join(OUT_DIR, "per_class_metrics.csv"), index=False)
print("\nPer-class metrics with prevalence "
      f"(classes in < {MIN_PAINTINGS} paintings are NOT resolved by this set):")
print(per_class.assign(IoU=per_class.IoU.round(2))[
      ["class", "IoU", "precision", "recall", "paintings_present", "resolved"]]
      .to_string(index=False))
unres = per_class.loc[~per_class.resolved, "class"].tolist()
if unres:
    print(f"\n  UNRESOLVED, do not report a number for these: {unres}")

if BENCHMARK_6:
    missing = [k for k in BENCHMARK_6 if k not in per_painting]
    if missing:
        print(f"\n[WARN] benchmark stems not found: {missing}")
    M6 = sum(per_painting[k] for k in BENCHMARK_6 if k in per_painting)
    i6, p6, r6 = metrics(M6)
    print(f"\nTable 1 comparability -- Mask2Former_ext on the {len(BENCHMARK_6)}-painting "
          "benchmark subset vs all 16:")
    cmp = pd.DataFrame({"class": NAMES,
                        "IoU_16": 100*iou, "IoU_6": 100*i6,
                        "diff": 100*(iou - i6)})
    print(cmp.round(2).to_string(index=False))
    print(f"\n  mIoU on 16 = {100*iou.mean():.2f} | mIoU on 6 = {100*i6.mean():.2f}")
    print("  Report the 6-painting figure in Table 1 against the other architectures,")
    print("  and the 16-painting figure wherever ext accuracy feeds the ecology.")
    cmp.round(2).to_csv(os.path.join(OUT_DIR, "table1_comparability.csv"), index=False)

# ------------------------------------------------------------------ bootstrap
B, rows = 2000, []
keys = list(per_painting)
rng = np.random.default_rng(42)
for _ in range(B):
    s = rng.choice(len(keys), len(keys), replace=True)
    mb = np.zeros((len(groups),)*2)
    Mb = sum(per_painting[keys[i]] for i in s)
    for a, ga in enumerate(groups):
        for b_, gb in enumerate(groups):
            mb[a, b_] = Mb[np.ix_(idx[ga], idx[gb])].sum()
    i_, p_, r_ = metrics(mb.astype(np.int64))
    rows.append(np.concatenate([i_, p_, r_]))
bs = np.array(rows)
lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
n = len(groups)
boot = pd.DataFrame({
    "part": groups,
    "IoU":        100*iou_p,  "IoU_lo":  lo[:n]*100,      "IoU_hi":  hi[:n]*100,
    "precision":  100*prec_p, "prec_lo": lo[n:2*n]*100,   "prec_hi": hi[n:2*n]*100,
    "recall":     100*rec_p,  "rec_lo":  lo[2*n:]*100,    "rec_hi":  hi[2*n:]*100})
boot.round(2).to_csv(os.path.join(OUT_DIR, "pooled_metrics_bootstrap.csv"), index=False)
print("\nPooled metrics with painting-level bootstrap CIs "
      f"(resampling {len(keys)} paintings):")
print(boot.round(1).to_string(index=False))
print("\nIf these intervals are very wide, that is the answer to 'is 6 paintings")
print("enough' -- it is not, and the k-fold route at the top of this file is the fix.")


Loading weights:   0%|          | 0/556 [00:00<?, ?it/s]

palette -> class mapping:
   0  (0, 0, 0)          background
   1  (110, 13, 13)      building
   2  (96, 66, 7)        earth
   3  (131, 224, 112)    grass
   4  (240, 120, 240)    animal
   5  (37, 70, 103)      mountain
   6  (230, 209, 168)    path_road
   7  (184, 61, 245)     person
   8  (65, 93, 125)      rock
   9  (48, 173, 48)      shrub_bush
  10  (18, 206, 242)     sky
  11  (13, 135, 53)      tree_conical
  12  (135, 246, 171)    tree_broadleaf
  13  (253, 164, 5)      wooded_mass
  14  (85, 144, 203)     Water
16 image/mask pairs
  1:    531200 px used | off-palette        0 | snapped        0 | dropped 0.000%
  10:    508800 px used | off-palette        0 | snapped        0 | dropped 0.000%
  100:    491200 px used | off-palette        0 | snapped        0 | dropped 0.000%
  1000:    462400 px used | off-palette        0 | snapped        0 | dropped 0.000%
  10000:    448800 px used | off-palette        0 | snapped        0 | dropped 0.000%
  100000:   1548400 px used 